# Feature Selection with Grey Wolf Optimization
Runs GWO/bGWO feature selection using the modular `src/gwo` package and inspects the result.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
import config
from src.preprocessing.load_dataset import load_train_test
from src.preprocessing.clean_data import clean_dataset
from src.preprocessing.encoding import encode_categorical_features, split_features_and_target
from src.gwo.feature_selector import GWOFeatureSelector
from src.models.model_factory import get_model_factory
from src.utils.constants import FEATURE_COLUMNS

In [ ]:
df_train, df_test = load_train_test(config.TRAIN_RAW_PATH, config.TEST_RAW_PATH)
df_train = clean_dataset(df_train)
df_test = clean_dataset(df_test)

X_train, y_train = split_features_and_target(df_train)
X_test, y_test = split_features_and_target(df_test)

X_train, X_test, _ = encode_categorical_features(X_train, X_test, config.CATEGORICAL_FEATURE_INDICES)
X_train.shape, X_test.shape

## Run GWO feature selection
(Reduce population/iterations here for a quick notebook run; use config.py values for the real experiment.)

In [ ]:
model_factory = get_model_factory(config.CLASSIFIER)

selector = GWOFeatureSelector(
    classifier_factory=model_factory,
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    categorical_indices=config.CATEGORICAL_FEATURE_INDICES,
    alpha=config.FITNESS_ALPHA, beta=config.FITNESS_BETA,
)

result = selector.run_multiple(
    variant=config.GWO_VARIANT,
    n_runs=config.GWO_RUNS,
    n_wolves=config.GWO_POPULATION_SIZE,
    max_iter=config.GWO_MAX_ITERATIONS,
)

print('Best accuracy:', result.best_wolf.score)
print('Best fitness:', result.best_wolf.fitness)
print('Selected features:', sum(result.best_wolf.features), '/', len(result.best_wolf.features))
print('AAP:', result.average_accuracy_performance)
print('ANF:', result.average_selected_features)

In [ ]:
import pandas as pd
pd.DataFrame({'feature': FEATURE_COLUMNS, 'selected': result.best_wolf.features})